# Memory Fundamentals

## Learning Objectives
By the end of this notebook you will be able to:
- Explain why memory is essential for LLM agents and long-running GenAI systems
- Distinguish working context, conversation history, and persistent memory
- Map the memory lifecycle: capture -> store -> retrieve -> update -> forget
- Identify failure modes when agents have no (or bad) memory

## Why AI Memory Matters

Large language models are **stateless** by default: each API call sees only the tokens you send. Without an external memory layer, an agent cannot:
- Recall user preferences across sessions
- Build a knowledge base from past tool results
- Avoid repeating failed actions
- Maintain project state over hours or days

Memory turns a chatbot into a system that *learns from interaction*.

### The Context Window Is Not Memory
The context window is temporary working space. It is limited, expensive, and discarded when the session ends. True memory outlives a single prompt and can be selectively retrieved.

## Core Concepts

### Working Memory vs Persistent Memory
| Layer | Lifetime | Typical store | Role |
|-------|----------|---------------|------|
| Working / short-term | Current turn or session | Prompt buffer, Redis TTL | Immediate reasoning |
| Long-term | Days-years | Vector DB, SQL, graph | Preferences, facts, episodes |
| Scratchpad | Single plan | In-agent state | Intermediate tool notes |

### Memory Lifecycle
1. **Capture** - decide what is worth remembering (facts, decisions, embeddings of dialogue)
2. **Store** - write to the right backend with metadata (user_id, timestamp, type)
3. **Retrieve** - query by similarity, filters, or graph traversal
4. **Inject** - place retrieved snippets into the prompt or agent state
5. **Update / Forget** - correct stale facts; prune low-value items

In [ ]:
# Conceptual memory lifecycle (no external services required)
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any
import hashlib

@dataclass
class MemoryItem:
    content: str
    memory_type: str  # fact | episode | preference | procedural
    user_id: str
    importance: float = 0.5
    metadata: dict[str, Any] = field(default_factory=dict)
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    @property
    def id(self) -> str:
        raw = f"{self.user_id}:{self.memory_type}:{self.content}"
        return hashlib.sha256(raw.encode()).hexdigest()[:16]

class InMemoryStore:
    """Tiny educational stand-in for a real memory backend."""
    def __init__(self):
        self._items: dict[str, MemoryItem] = {}

    def store(self, item: MemoryItem) -> str:
        self._items[item.id] = item
        return item.id

    def retrieve(self, query: str, user_id: str, top_k: int = 3) -> list[MemoryItem]:
        # Naive keyword score - real systems use embeddings + filters
        q = set(query.lower().split())
        scored = []
        for item in self._items.values():
            if item.user_id != user_id:
                continue
            overlap = len(q & set(item.content.lower().split()))
            score = overlap + item.importance
            if score > 0:
                scored.append((score, item))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [item for _, item in scored[:top_k]]

store = InMemoryStore()
store.store(MemoryItem(
    content="User prefers concise bullet answers and UTC timestamps.",
    memory_type="preference",
    user_id="u_42",
    importance=0.9,
))
store.store(MemoryItem(
    content="Project Phoenix uses PostgreSQL and prefers Qdrant for vectors.",
    memory_type="fact",
    user_id="u_42",
    importance=0.8,
))

hits = store.retrieve("What database does Phoenix use?", user_id="u_42")
for h in hits:
    print(f"[{h.memory_type}] {h.content}")

## Memory in Agent Architectures

```
User -> Agent Planner -> Tools
              ↓
         Memory Layer
         (retrieve before act, write after act)
```

Common patterns:
- **Retrieve-then-generate**: fetch memories, then call the LLM
- **Write-through**: after each turn, extract facts and persist
- **Reflective memory**: periodic summarization of episodes into semantic facts

### Design Principles
1. **Scoped by identity** - always key by `user_id` / `tenant_id` / `session_id`
2. **Typed memories** - preferences ≠ tool traces ≠ documents
3. **Provenance** - store source message IDs for auditability
4. **Budget awareness** - retrieval must fit the token budget

In [ ]:
# Prompt injection pattern with a memory budget
def build_prompt_with_memory(
    system: str,
    user_message: str,
    memories: list[str],
    max_memory_chars: int = 800,
) -> str:
    block = []
    used = 0
    for m in memories:
        if used + len(m) > max_memory_chars:
            break
        block.append(f"- {m}")
        used += len(m)
    memory_section = "\n".join(block) if block else "(none)"
    return f"""{system}

## Retrieved Memory
{memory_section}

## User
{user_message}
"""

prompt = build_prompt_with_memory(
    system="You are a helpful project assistant.",
    user_message="Remind me of our stack choices.",
    memories=[h.content for h in hits],
)
print(prompt)

## Failure Modes Without Good Memory
- **Amnesia**: agent re-asks the same questions every session
- **Contradiction**: old and new facts both retrieved without conflict resolution
- **Privacy leaks**: wrong tenant's memories retrieved
- **Context bloat**: dumping entire history instead of selective recall
- **Stale procedures**: outdated runbooks kept as "truth"

## Key Takeaways
- LLMs need an explicit memory subsystem beyond the context window
- Separate capture, storage, retrieval, and forgetting as first-class operations
- Scope, type, and provenance are as important as the embedding model
- Start simple (structured facts + vector search) before complex agent memory stacks